# Seka Kama — Scenario Analysis Notebook

Use this notebook to:
- Load the SekaNet XGBoost model and inspect feature importances
- Run what-if scenarios offline (without the FastAPI server)
- Visualise spatial outputs using GeoPandas & Matplotlib
- Export results for import into Kepler.gl

**Prerequisites:** activate your Python venv and run `pip install -r ../requirements.txt`

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import joblib
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path
from dotenv import load_dotenv

load_dotenv('../.env')
print('Environment loaded.')

## 1. Load Model Artefacts

In [ ]:
MODELS_DIR = Path('../../web-app/models')

model         = joblib.load(MODELS_DIR / 'sekanet_xgboost_shp.pkl')
scaler        = joblib.load(MODELS_DIR / 'sekanet_scaler_shp.pkl')
feature_names = joblib.load(MODELS_DIR / 'feature_names.pkl')

print(f'Model loaded. Features: {len(feature_names)}')
print('Feature list (first 10):', feature_names[:10])

## 2. Feature Importances

In [ ]:
importance = pd.DataFrame({
    'feature': feature_names,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

top_n = importance.head(15)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_n['feature'][::-1], top_n['importance'][::-1], color='#4CAF50')
ax.set_xlabel('Feature Importance')
ax.set_title('SekaNet XGBoost — Top 15 Predictors')
plt.tight_layout()
plt.show()
importance.head(15)

## 3. Load Grid Cells from Supabase

In [ ]:
from supabase import create_client

sb = create_client(os.environ['SUPABASE_URL'], os.environ['SUPABASE_SERVICE_ROLE_KEY'])

result = sb.table('grid_cells').select(
    'cell_id, management_unit, baseline_lion_density, ' + ','.join(feature_names[:10])
).limit(5000).execute()

df = pd.DataFrame(result.data)
print(f'Loaded {len(df)} cells. Columns: {list(df.columns[:8])}')
df.head()

## 4. Run a What-If Scenario

In [ ]:
from services.prediction_service import PredictionService

svc = PredictionService(model, scaler, feature_names)

# Example: increase nightlight trend by 20% across all cells
modifications = {'longterm_slope_mean': 0.20}

cells = df.to_dict('records')
results = svc.calculate_scenario_impact(cells, modifications, apply_percent=True)

print(f"Baseline total: {results['baseline_total']:.1f} lions")
print(f"Scenario total: {results['scenario_total']:.1f} lions")
print(f"Delta:          {results['delta_total']:+.1f} lions ({results['delta_percent_total']:+.1f}%)")

## 5. Visualise Per-Unit Impacts

In [ ]:
unit_data = pd.DataFrame([
    {'unit': k, **v} for k, v in results['unit_aggregation'].items()
]).sort_values('delta', ascending=True)

colors = ['#e53935' if d < 0 else '#43a047' for d in unit_data['delta']]

fig, ax = plt.subplots(figsize=(10, max(4, len(unit_data) * 0.4)))
ax.barh(unit_data['unit'], unit_data['delta'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Change in Lion Abundance')
ax.set_title('Per-Conservancy Impact')
plt.tight_layout()
plt.show()